In [0]:
CREATE OR REPLACE VIEW sac.customer.ai_address AS
SELECT
	customer_id,
	street,
	city,
	ai_query(
		"databricks-gemma-3-12b",
		request =>
			concat(
				"Gebe basierend auf einem Gemeindenamen als Ergebnis eine JSON Array aus, das die zugehörige Postleitzahl (PLZ) und das zugehörige Bundesland aus der Json Enumeration enthält. Es dürfen keine anderen Bundesländer genannt werden. Wenn das Ergebnis unbekannt ist, gebe kein Wert als Ergebnis. Bei dem übergebenen Wert city handelt es sich um die Gemeinde. Suche nach der PLZ immer in diesem Format: '[city] plz' und basierend auf der PLZ nach dem Bundesland. Die Gemeinde Königsee und Frankenblick gehört beispielsweise zum Bundesland Thüringen, obwohl es diese Sehenswürdigkeiten in Bayern gibt. Auch Treben, Unstruttal und Harztor sind Gemeinden in Thüringen.

Bundesland-Liste: Schleswig-Holstein, Hamburg, Sachsen, Berlin, Thüringen, Brandenburg, Nordrhein-Westfalen


Beispiel:

street
Über der Mühle 3
city
Ilmtal-Weinstraße

RESULT
[
{'plz': '99510','bundeland': 'Thüringen'}
]

DOCUMENT\n",
				city,
				'\n\nRESULT\n'
			),
		responseFormat =>
			'{
              "type": "json_schema",
              "json_schema": {
                  "name": "address_schema",
                  "schema": {
                      "type": "array",
                      "items": {
                          "type": "object",
                          "properties": {
                              "plz": { 
																"type": "string"
															},
                              "bundesland": { 
																"type": "string",
                                "enum": ["Schleswig-Holstein", "Hamburg", "Sachsen", "Berlin", "Thueringen", "Brandenburg", "Nordrhein-Westfalen"] 
															}
                          }
                      }
                  },
                  "strict": true
              }
          }'
	) AS address
FROM
	sac.customer.customer
WHERE
	plz = ''
	OR plz IS NULL;

-- Add values to missing adresses
MERGE INTO
	sac.customer.customer c
USING (
	SELECT
		customer_id,
		left(ai.plz, 5) AS plz,
		CASE ai.bundesland
			WHEN 'Thueringen' THEN 'Thüringen'
			ELSE ai.bundesland
		END AS bundesland 
	FROM
		sac.customer.ai_address aia
		LATERAL VIEW OUTER EXPLODE(from_json(aia.address, 'ARRAY<MAP<STRING,STRING>>')) AS ai
	WHERE
		ai.plz IS NOT NULL
		AND ai.plz != ''
		AND ai.plz != lower('null')
	QUALIFY
		row_number() OVER (PARTITION BY customer_id ORDER BY ai.plz) = 1
) a
ON
	c.customer_id = a.customer_id
WHEN MATCHED THEN UPDATE SET
	c.plz = a.plz, c.zone = a.bundesland, c.ai_used = CAST('true' AS BOOLEAN)